# 03f — Two-Stage (lgbm) 임시 단일 노트북 — **unit aggregate 버전**

**목적**: 03b의 die-level broadcast 대신 **die→unit 집계 후 unit-level 직접 학습**니 패턴으로 한 번 돌려서 plateau에 들어오는지 확인. **일회용**.

**근거**: 1차 baseline `30-931-001` (`optuna_merged.db` study) best trial #2204
- 해당 study가 unit aggregation 계열에서 가장 좋은 RMSE (study 내 metric 기준)
- agg_funcs = `[mean, std, range, min, max, median]` 6종 전부 사용
- HP가 final 평가 metric에서 best임을 보장은 못하지만, unit-agg 계열 파일럿으로는 가장 근거 있는 시작점.

**구성**:
- 전처리 = `30-931-001` cleaning_args (↑), `corr_keep_by`는 leakage 위험도 때문에 `target_corr` → **`std`** 로 안전 변경 (사용자 동의)
- die→unit 집계: `utils.aggregate.aggregate_to_unit(agg_funcs=6종)`
- Stage 1 (분류): LGBMClassifier(objective='binary'), HP = 30-931-001 best (objective만 binary로 override)
- Stage 2 (회귀): LGBMRegressor(objective='poisson'), HP = 30-931-001 best, **target = log1p(y_unit), y_unit > 0 unit만**
- 최종 unit pred = `P(y>0) × expm1(reg_log)` — unit-level 직접 곱, **집계 추가 X**

**격리**: `4_output/_temp/two_stage_unit_agg/` 신규. 모듈 무수정.

**비교 대상**:
- 03b (die-level broadcast, log1p preset PP, mean agg): val=0.005718, test=0.008417
- reg_only/lgbm 단독: val=0.005731, test=0.008429
- BagZIT plateau best: val=0.005701, test=0.008408
- 본 노트북이 03b를 넘으면 unit aggregation 패턴이 die-level broadcast보다 우세함

## 1. 환경 + import

In [2]:
import os, sys, json

%run ../../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import (
    PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, OUTPUT_DIR,
)
from utils.data import load_all, get_feat_cols, split_xs
from utils.aggregate import aggregate_to_unit

MODEL_ROOT = os.path.join(PROJECT_ROOT, '3_modeling')
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)

from final.modules import preprocess

import lightgbm as lgb
from sklearn.model_selection import KFold

import logging
logging.getLogger('lightgbm').setLevel(logging.ERROR)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')

setup 완료
PROJECT_ROOT = c:\Users\Dell5371\Desktop\기업연계프로젝트


## 2. 설정 (30-931-001 best PP/HP, log1p, unit aggregate 6종)

1차 baseline `30-931-001` study best trial #2204 추출 결과:
- **PP**: missing=0.9, corr=0.98, corr_keep_by='std' (`target_corr` → `std` 안전 변경, leakage 회피)
- **HP (LGBM)**: lr=0.00553, n_est=903, num_leaves=178, depth=12, min_child=142, colsample=0.674, subsample=0.611, reg_α=0.00126, reg_λ=3.1e-7, min_split_gain=2.1e-9, path_smooth=16.06
- **agg_funcs**: mean / std / range / min / max / median (6종)
- objective: 회귀=poisson (Stage 2), 분류=binary (Stage 1)

In [3]:
EXP_ID = 'two-stage-unit-agg-001'
N_FOLDS = 5
CLIP_Y_EXTREME = True
TARGET_TRANSFORM = 'log1p'

OUT_DIR = os.path.join(OUTPUT_DIR, '_temp', 'two_stage_unit_agg')
os.makedirs(OUT_DIR, exist_ok=True)

# ── 전처리 PARAMS (30-931-001 cleaning_args, target_corr→std 안전 변경) ──
PARAMS = {
    'missing_threshold':          0.9,
    'corr_threshold':             0.98,
    'corr_keep_by':               'std',     # ★ 30-931-001은 'target_corr' (leakage) → 'std' 로 변경
    'add_indicator':              False,
    'indicator_threshold':        0.15,
    'spatial_max_dist':           1.0,
    'post_impute_corr_threshold': 0.98,
    'post_impute_corr_keep_by':   'std',
}

# ── 집계 함수 6종 ──
AGG_FUNCS = ['mean', 'std', 'range', 'min', 'max', 'median']

# ── LGBM HP (30-931-001 best trial #2204 trial_params 그대로) ──
LGB_HP_BASE = dict(
    n_estimators       = 903,
    learning_rate      = 0.005533573630889979,
    num_leaves         = 178,
    max_depth          = 12,
    min_child_samples  = 142,
    subsample          = 0.6105634056703496,
    subsample_freq     = 1,
    colsample_bytree   = 0.6738919590642785,
    reg_alpha          = 0.0012579407169105439,
    reg_lambda         = 3.1133747938697414e-07,
    min_split_gain     = 2.1083999800006533e-09,
    path_smooth        = 16.0602712938438,
    random_state       = SEED,
    n_jobs             = -1,
    verbose            = -1,
)
REG_OBJECTIVE = 'poisson'   # Stage 2 — 03b와 동일
CLF_OBJECTIVE = 'binary'    # Stage 1

print(f'EXP_ID={EXP_ID} | N_FOLDS={N_FOLDS}')
print(f'TARGET_TRANSFORM={TARGET_TRANSFORM}')
print(f'OUT_DIR={OUT_DIR}')
print(f'PARAMS keys: {list(PARAMS)}')
print(f'AGG_FUNCS: {AGG_FUNCS}')
print(f'LGB_HP_BASE: {len(LGB_HP_BASE)} keys | reg objective={REG_OBJECTIVE}, clf objective={CLF_OBJECTIVE}')

EXP_ID=two-stage-unit-agg-001 | N_FOLDS=5
TARGET_TRANSFORM=log1p
OUT_DIR=c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\two_stage_unit_agg
PARAMS keys: ['missing_threshold', 'corr_threshold', 'corr_keep_by', 'add_indicator', 'indicator_threshold', 'spatial_max_dist', 'post_impute_corr_threshold', 'post_impute_corr_keep_by']
AGG_FUNCS: ['mean', 'std', 'range', 'min', 'max', 'median']
LGB_HP_BASE: 15 keys | reg objective=poisson, clf objective=binary


## 3. 데이터 로드 + Y clip

In [4]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)

ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개 샘플')

y_train_unit = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_unit   = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_unit  = ys_input['test'].set_index(KEY_COL)[TARGET_COL]

print(f'\n[데이터 로드] xs={xs.shape}, X feat_cols={len(feat_cols)}')
print(f'  unit train={len(y_train_unit):,}, val={len(y_val_unit):,}, test={len(y_test_unit):,}')
print(f'  y_train: max={y_train_unit.max():.6f}, mean={y_train_unit.mean():.6f}, zero ratio={(y_train_unit==0).mean():.1%}')

[load_xs] all-NaN 행 407개 제거 → 174,573행
[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572
[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
[CLIP_Y_EXTREME] 1.0 → 0.097417 clip, 1개 샘플

[데이터 로드] xs=(174572, 1091), X feat_cols=1087
  unit train=26,187, val=8,727, test=8,729
  y_train: max=0.097417, mean=0.002481, zero ratio=70.8%


## 4. 전처리 (30-931-001 cleaning_args)

die-level cleaning. 이 단계에서는 아직 die-level 유지.

In [5]:
pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PARAMS)
xs_train_die = pp['xs_train']
xs_val_die   = pp['xs_val']
xs_test_die  = pp['xs_test']
feat_cols_clean = pp['feat_cols']

print(f'\n[cleaning] feat_cols_clean={len(feat_cols_clean)}')
print(f'  xs_train_die: {xs_train_die.shape}, val: {xs_val_die.shape}, test: {xs_test_die.shape}')

[Stage 0] 웨이퍼맵 사전 제외: 1087 → 1033 (54개 제거)
클리닝 파이프라인 시작
원본 feature 수: 1033
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 928개
    컬럼: 1033 → 928 (105개 제거)
    DataFrame: (104748, 986)

[고결측 제거] threshold=90%
  제거: 2개, 잔여: 926개
    컬럼: 928 → 926 (2개 제거)
    DataFrame: (104748, 984)

[중복 컬럼 제거] sample_n=5000
  제거: 27개, 잔여: 899개
    컬럼: 926 → 899 (27개 제거)
    DataFrame: (104748, 957)

[고상관 제거] threshold=0.98, keep_by=std (std)
  제거: 136개, 잔여: 763개
    컬럼: 899 → 763 (136개 제거)
    DataFrame: (104748, 821)

[공간 보간 imputation] 총 결측: 753,906
  train-only 모드: train 104,748 / 전체 174,572 행
  1단계 (공간 보간, dist<=1.0): 53,921개 채움 → 잔여: 699,985
  2단계 (lot 평균, train 기준): 509,759개 채움 → 잔여: 190,226
  3단계 (train 전체 평균): 190,226개 채움 → 잔여: 0

  [요약] 753,906 → 공간(53,921) → lot(509,759) → 전체(190,226) → 잔여(0)

[고상관 제거] threshold=0.98, keep_by=std (std)
  제거: 0개, 잔여: 763개
    [고상관 제거 2차 / imputation 후] threshold=0.98
    컬럼: 763 → 763 (0개 제거)
    DataFrame: (104748, 821)

클리닝 완료: 1033 → 763 features (270개 제거)
  

## 5. die → unit 집계 (6종)

`utils.aggregate.aggregate_to_unit` 사용. mean/std/range/min/max/median → feat_cols × 6 개 확장.

In [6]:
X_train_unit_df = aggregate_to_unit(xs_train_die, feat_cols=feat_cols_clean, agg_funcs=AGG_FUNCS)
X_val_unit_df   = aggregate_to_unit(xs_val_die,   feat_cols=feat_cols_clean, agg_funcs=AGG_FUNCS)
X_test_unit_df  = aggregate_to_unit(xs_test_die,  feat_cols=feat_cols_clean, agg_funcs=AGG_FUNCS)

# 결측 체크 (range/std/median 이 NaN 이 될 수 있는지)
n_nan_train = X_train_unit_df.isna().sum().sum()
n_nan_val   = X_val_unit_df.isna().sum().sum()
n_nan_test  = X_test_unit_df.isna().sum().sum()
if n_nan_train + n_nan_val + n_nan_test > 0:
    print(f'\n[⚠ NaN 검출] train={n_nan_train}, val={n_nan_val}, test={n_nan_test} — 0으로 체우기')
    X_train_unit_df = X_train_unit_df.fillna(0.0)
    X_val_unit_df   = X_val_unit_df.fillna(0.0)
    X_test_unit_df  = X_test_unit_df.fillna(0.0)

# y 와 정렬 (index = ufs_serial 기준)
X_train = X_train_unit_df.loc[y_train_unit.index].values.astype(np.float64)
X_val   = X_val_unit_df.loc[y_val_unit.index].values.astype(np.float64)
X_test  = X_test_unit_df.loc[y_test_unit.index].values.astype(np.float64)
y_train = y_train_unit.values.astype(np.float64)
y_val   = y_val_unit.values.astype(np.float64)
y_test  = y_test_unit.values.astype(np.float64)
y_train_bin = (y_train > 0).astype(np.int32)

unit_feat_names = list(X_train_unit_df.columns)
print(f'\n[집계 완료] unit_feat_names={len(unit_feat_names)} (= {len(feat_cols_clean)} × {len(AGG_FUNCS)})')
print(f'  X_train: {X_train.shape}, val: {X_val.shape}, test: {X_test.shape}')
print(f'  y_train_bin pos ratio={y_train_bin.mean():.4f}')

집계 완료: 26,187 units × 4,578 features (agg: ['mean', 'std', 'range', 'min', 'max', 'median'])
집계 완료: 8,727 units × 4,578 features (agg: ['mean', 'std', 'range', 'min', 'max', 'median'])
집계 완료: 8,729 units × 4,578 features (agg: ['mean', 'std', 'range', 'min', 'max', 'median'])

[집계 완료] unit_feat_names=4578 (= 763 × 6)
  X_train: (26187, 4578), val: (8727, 4578), test: (8729, 4578)
  y_train_bin pos ratio=0.2920


## 6. KFold split (unit-level 직접)

die-level mask 필요 없음. `KFold(N_FOLDS, shuffle=True, random_state=SEED)` 으로 y_train_unit 인덱스 장축 직접 분할.

In [7]:
kf_global = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(kf_global.split(np.arange(len(y_train))))
print(f'fold split: {N_FOLDS} folds')
for fi, (tr_idx, vl_idx) in enumerate(FOLDS):
    print(f'  fold {fi+1}: train={len(tr_idx):,} val={len(vl_idx):,}')

fold split: 5 folds
  fold 1: train=20,949 val=5,238
  fold 2: train=20,949 val=5,238
  fold 3: train=20,950 val=5,237
  fold 4: train=20,950 val=5,237
  fold 5: train=20,950 val=5,237


## 7. 5-fold Two-Stage 학습 (unit-level 직접)

각 fold:
1. **Stage 1 분류**: LGBMClassifier(objective='binary'), target = `(y_unit > 0)`, 모든 train unit
2. **Stage 2 회귀**: LGBMRegressor(objective='poisson'), target = `log1p(y_unit)`, **y_unit > 0 unit만**
3. **최종 unit pred** = `prob_unit × expm1(reg_log_unit)`

die→unit 추가 집계 없음 (이미 unit 레벨).

In [8]:
import time

n_train_unit = len(X_train)
n_val_unit   = len(X_val)
n_test_unit  = len(X_test)

# unit-level 캐쳐
oof_unit_prob   = np.full(n_train_unit, np.nan)
oof_unit_reg    = np.full(n_train_unit, np.nan)
oof_unit_pred   = np.full(n_train_unit, np.nan)

val_unit_prob   = np.zeros(n_val_unit)
val_unit_reg    = np.zeros(n_val_unit)
val_unit_pred   = np.zeros(n_val_unit)

test_unit_prob  = np.zeros(n_test_unit)
test_unit_reg   = np.zeros(n_test_unit)
test_unit_pred  = np.zeros(n_test_unit)

print('=== 5-fold Two-Stage 학습 (unit-level) ===')
t0 = time.time()
for fold_idx, (tr_idx, vl_idx) in enumerate(FOLDS):
    X_tr  = X_train[tr_idx]
    X_vl  = X_train[vl_idx]
    y_tr  = y_train[tr_idx]
    yb_tr = y_train_bin[tr_idx]

    # ── Stage 1 분류 (LGBMClassifier, binary) ──
    clf = lgb.LGBMClassifier(**LGB_HP_BASE, objective=CLF_OBJECTIVE)
    clf.fit(X_tr, yb_tr)

    # ── Stage 2 회귀 (y>0 unit만, log1p target, poisson objective) ──
    pos_mask = y_tr > 0
    if pos_mask.sum() < 100:
        raise RuntimeError(f'fold {fold_idx+1}: y>0 unit 수가 너무 적음 ({pos_mask.sum()})')
    y_tr_pos_log = np.log1p(y_tr[pos_mask])
    reg = lgb.LGBMRegressor(**LGB_HP_BASE, objective=REG_OBJECTIVE)
    reg.fit(X_tr[pos_mask], y_tr_pos_log)

    # ── 예측 ──
    def _predict(Xs):
        prob = clf.predict_proba(Xs)[:, 1]
        prob = np.clip(prob, 0.0, 1.0)
        reg_log = reg.predict(Xs)
        reg_y   = np.clip(np.expm1(reg_log), 0.0, None)
        final   = prob * reg_y
        return prob, reg_y, final

    p_vl, r_vl, f_vl = _predict(X_vl)
    p_v,  r_v,  f_v  = _predict(X_val)
    p_t,  r_t,  f_t  = _predict(X_test)

    # OOF (vl)
    oof_unit_prob[vl_idx] = p_vl
    oof_unit_reg[vl_idx]  = r_vl
    oof_unit_pred[vl_idx] = f_vl

    # val/test 5-fold avg
    val_unit_prob  += p_v / N_FOLDS
    val_unit_reg   += r_v / N_FOLDS
    val_unit_pred  += f_v / N_FOLDS
    test_unit_prob += p_t / N_FOLDS
    test_unit_reg  += r_t / N_FOLDS
    test_unit_pred += f_t / N_FOLDS

    print(f'  fold {fold_idx+1}/{N_FOLDS} done ({time.time()-t0:.0f}s) — '
          f'pos unit ratio={pos_mask.mean():.3f}, prob_vl mean={p_vl.mean():.4f}, '
          f'reg_vl mean={r_vl.mean():.6f}')

assert not np.isnan(oof_unit_prob).any(), 'OOF unit prob 미커버'
assert not np.isnan(oof_unit_reg).any(),  'OOF unit reg 미커버'
assert not np.isnan(oof_unit_pred).any(), 'OOF unit pred 미커버'

print(f'\n[학습 완료] {time.time()-t0:.0f}s')

=== 5-fold Two-Stage 학습 (unit-level) ===
  fold 1/5 done (70s) — pos unit ratio=0.290, prob_vl mean=0.2835, reg_vl mean=0.008404
  fold 2/5 done (144s) — pos unit ratio=0.294, prob_vl mean=0.2849, reg_vl mean=0.008420
  fold 3/5 done (264s) — pos unit ratio=0.293, prob_vl mean=0.2853, reg_vl mean=0.008412
  fold 4/5 done (377s) — pos unit ratio=0.292, prob_vl mean=0.2869, reg_vl mean=0.008472
  fold 5/5 done (507s) — pos unit ratio=0.291, prob_vl mean=0.2854, reg_vl mean=0.008410

[학습 완료] 507s


## 8. RMSE 평가

In [9]:
def _rmse(pred, true):
    return float(np.sqrt(np.mean((np.asarray(pred) - np.asarray(true)) ** 2)))

oof_rmse  = _rmse(oof_unit_pred,  y_train)
val_rmse  = _rmse(val_unit_pred,  y_val)
test_rmse = _rmse(test_unit_pred, y_test)

print('=' * 75)
print(f'  Two-Stage unit-agg (lgbm, 30-931-001 PP/HP, log1p, agg=6종)')
print('=' * 75)
print(f'  {"":12s}  {"OOF":>11s}  {"val":>11s}  {"test":>11s}')
print(f'  {"unit RMSE":12s}  {oof_rmse:11.6f}  {val_rmse:11.6f}  {test_rmse:11.6f}')
print('-' * 75)
print(f'  비교 기준선:')
print(f'    03b (die-level broadcast):        val=0.005718, test=0.008417')
print(f'    reg_only/lgbm (Stage 1 없음):     val=0.005731, test=0.008429')
print(f'    BagZIT plateau best (zit_only):   val=0.005709, test=0.008414')
print(f'    Stacking 11-base (val best):       val=0.005701, test=0.008408')
print('=' * 75)
if val_rmse < 0.0058:
    print('  → plateau 영역 안. stacking pool 추가 후보.')
elif val_rmse < 0.006:
    print('  → plateau 근처. residual 패턴이 다르면 stacking에 도움 가능.')
else:
    print('  → plateau 밖. unit aggregation 패턴이 die-level보다 불리.')

  Two-Stage unit-agg (lgbm, 30-931-001 PP/HP, log1p, agg=6종)
                        OOF          val         test
  unit RMSE        0.005530     0.005742     0.008438
---------------------------------------------------------------------------
  비교 기준선:
    03b (die-level broadcast):        val=0.005718, test=0.008417
    reg_only/lgbm (Stage 1 없음):     val=0.005731, test=0.008429
    BagZIT plateau best (zit_only):   val=0.005709, test=0.008414
    Stacking 11-base (val best):       val=0.005701, test=0.008408
  → plateau 영역 안. stacking pool 추가 후보.


## 9. 산출물 저장 (`_temp/two_stage_unit_agg/`)

In [10]:
def _build_unit_df(uid, prob, reg, pred, y_true):
    return pd.DataFrame({
        KEY_COL: uid,
        'prob':  prob,
        'reg':   reg,
        'pred':  pred,
        TARGET_COL: y_true,
    })

_build_unit_df(y_train_unit.index.values, oof_unit_prob,  oof_unit_reg,  oof_unit_pred,  y_train) \
    .to_csv(os.path.join(OUT_DIR, 'oof_unit.csv'),  index=False)
_build_unit_df(y_val_unit.index.values,   val_unit_prob,  val_unit_reg,  val_unit_pred,  y_val) \
    .to_csv(os.path.join(OUT_DIR, 'val_unit.csv'),  index=False)
_build_unit_df(y_test_unit.index.values,  test_unit_prob, test_unit_reg, test_unit_pred, y_test) \
    .to_csv(os.path.join(OUT_DIR, 'test_unit.csv'), index=False)

meta = {
    'exp_id':            EXP_ID,
    'model':             'Two-Stage unit-agg (lgbm clf binary + lgbm reg poisson) + log1p + 6 agg',
    'target_transform':  TARGET_TRANSFORM,
    'aggregation':       AGG_FUNCS,
    'training_level':    'unit (post-aggregate)',
    'n_folds':           N_FOLDS,
    'oof_rmse':          oof_rmse,
    'val_rmse':          val_rmse,
    'test_rmse':         test_rmse,
    'preprocess_PARAMS': PARAMS,
    'effective_pp_params': pp['effective_params'],
    'lgb_hp_base':       {k: v for k, v in LGB_HP_BASE.items() if k not in ['random_state', 'n_jobs', 'verbose']},
    'reg_objective':     REG_OBJECTIVE,
    'clf_objective':     CLF_OBJECTIVE,
    'pp_source':         '30-931-001 best trial #2204 cleaning_args (corr_keep_by: target_corr → std 안전 변경)',
    'hp_source':         '30-931-001 best trial #2204 trial_params',
    'CLIP_Y_EXTREME':    CLIP_Y_EXTREME,
    'feat_cols_clean_n': len(feat_cols_clean),
    'unit_feat_n':       len(unit_feat_names),
    'SEED':              int(SEED),
}
with open(os.path.join(OUT_DIR, 'meta.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False, default=str)

print(f'저장 완료: {OUT_DIR}')
for f_ in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f_)) / 1024
    print(f'  {f_:25s}  {sz:>10,.1f} KB')

저장 완료: c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\two_stage_unit_agg
  meta.json                         2.0 KB
  oof_unit.csv                  2,001.4 KB
  test_unit.csv                   667.1 KB
  val_unit.csv                    666.8 KB


## 10. 요약

In [11]:
print('=' * 75)
print(f' Two-Stage unit-agg (lgbm) — 결과 요약')
print('=' * 75)
print(f'  EXP_ID            : {EXP_ID}')
print(f'  PP source         : 30-931-001 best trial #2204 (target_corr→std 변경)')
print(f'  HP source         : 30-931-001 best trial #2204 (objective: poisson)')
print(f'  target transform  : {TARGET_TRANSFORM}')
print(f'  agg funcs         : {AGG_FUNCS} ({len(AGG_FUNCS)}종)')
print(f'  feat_cols (clean) : {len(feat_cols_clean)} (die-level)')
print(f'  unit_feat (post-agg): {len(unit_feat_names)} (= {len(feat_cols_clean)} × {len(AGG_FUNCS)})')
print('-' * 75)
print(f'  {"":10s}  {"OOF":>11s}  {"val":>11s}  {"test":>11s}')
print(f'  {"unit RMSE":10s}  {oof_rmse:11.6f}  {val_rmse:11.6f}  {test_rmse:11.6f}')
print('-' * 75)
print(f'  → 03b (die-level broadcast, val=0.005718) 대비 개선되면 unit aggregation 우세.')
print(f'  → plateau(0.0057x/0.0084x) 안에 들어오면 stacking 후보.')
print(f'  → residual corr 가 03b/BagZIT 변형들과 다르면 stacking 효과 기대.')
print('=' * 75)

 Two-Stage unit-agg (lgbm) — 결과 요약
  EXP_ID            : two-stage-unit-agg-001
  PP source         : 30-931-001 best trial #2204 (target_corr→std 변경)
  HP source         : 30-931-001 best trial #2204 (objective: poisson)
  target transform  : log1p
  agg funcs         : ['mean', 'std', 'range', 'min', 'max', 'median'] (6종)
  feat_cols (clean) : 763 (die-level)
  unit_feat (post-agg): 4578 (= 763 × 6)
---------------------------------------------------------------------------
                      OOF          val         test
  unit RMSE      0.005530     0.005742     0.008438
---------------------------------------------------------------------------
  → 03b (die-level broadcast, val=0.005718) 대비 개선되면 unit aggregation 우세.
  → plateau(0.0057x/0.0084x) 안에 들어오면 stacking 후보.
  → residual corr 가 03b/BagZIT 변형들과 다르면 stacking 효과 기대.
